# Step 0: Explore listings.csv, choose columns, create the sample

Goal of this notebook (This is nootebook for recognition of data):
1. Load the raw file and check that it is parsed correctly
2. Look at the schema and choose the columns we keep
3. Measure nulls and look at raw values, to fill in `data_contract.md`
4. Create a 5,000-row sample for fast work

Expected location: `notebooks/scratch/00_step0_explore.ipynb`, with the data in `data/raw/listings.csv`.

In [1]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("airbnb-step0")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

DATA_PATH = "../../data/raw/listings.csv"
SAMPLE_DIR = "../../data/sample"
SAMPLE_SIZE = 5000
SEED = 42

assert os.path.exists(DATA_PATH), f"File not found: {os.path.abspath(DATA_PATH)}"


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 21:48:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Load and check parsing
Descriptions contain line breaks and doubled quotes, so `multiLine` and `escape` are required.
Everything is read as text on purpose: we decide the types ourselves during cleaning.

In [2]:
raw = (spark.read
       .option("header", True)
       .option("multiLine", True)
       .option("escape", '"')
       .csv(DATA_PATH))

n_rows = raw.count()
print(f"Rows: {n_rows:,} | Columns: {len(raw.columns)}")

# If parsing broke, pieces of descriptions end up in the id column.
bad_ids = raw.filter(~F.col("id").rlike("^[0-9]+$")).count()
print("Rows with a non-numeric id (should be 0):", bad_ids)

Rows: 77,149 | Columns: 90


[Stage 4:>                                                          (0 + 1) / 1]

Rows with a non-numeric id (should be 0): 0


## 2. All columns

In [3]:
for i, c in enumerate(raw.columns):
    print(f"{i:3d}  {c}")

  0  id
  1  listing_url
  2  scrape_id
  3  last_scraped
  4  source
  5  name
  6  description
  7  neighborhood_overview
  8  picture_url
  9  host_id
 10  host_url
 11  host_profile_id
 12  host_profile_url
 13  host_name
 14  host_since
 15  hosts_time_as_user_years
 16  hosts_time_as_user_months
 17  hosts_time_as_host_years
 18  hosts_time_as_host_months
 19  host_location
 20  host_about
 21  host_response_time
 22  host_response_rate
 23  host_acceptance_rate
 24  host_is_superhost
 25  host_thumbnail_url
 26  host_picture_url
 27  host_neighbourhood
 28  host_listings_count
 29  host_total_listings_count
 30  host_verifications
 31  host_has_profile_pic
 32  host_identity_verified
 33  neighbourhood
 34  neighbourhood_cleansed
 35  neighbourhood_group_cleansed
 36  latitude
 37  longitude
 38  property_type
 39  room_type
 40  accommodates
 41  bathrooms
 42  bathrooms_text
 43  bedrooms
 44  beds
 45  amenities
 46  price
 47  price_quote_checkin_date
 48  price_quote_checko

In [4]:
rows = raw.limit(200).collect()

for i, c in enumerate(raw.columns):
    examples = []
    for r in rows:
        v = r[c]
        if v is not None and v not in examples:
            examples.append(v)
        if len(examples) == 3:
            break
    shown = [str(v)[:60] for v in examples]
    print(f"{i:3d}  {c:45s} {shown}")

26/11/26 21:48:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  0  id                                            ['3109', '7397', '9952']
  1  listing_url                                   ['https://www.airbnb.com/rooms/3109', 'https://www.airbnb.com/rooms/7397', 'https://www.airbnb.com/rooms/9952']
  2  scrape_id                                     ['20260916052215']
  3  last_scraped                                  ['2026-09-19', '2026-09-22', '2026-09-21']
  4  source                                        ['previous scrape']
  5  name                                          ['zen and calm', 'MARAIS - 2ROOMS APT - 2/4 PEOPLE', 'Paris petit coin douillet']
  6  description                                   ['Lovely Appartment with one bedroom with a Queen size bed.<br', 'VERY CONVENIENT, WITH THE BEST LOCATION !', 'I am a retired lady, who offers a pleasant little apartment ']
  7  neighborhood_overview                         []
  8  picture_url                                   ['https://a0.muscache.com/pictures/miso/Hosting-3109/original/'

## 3. Candidate columns
Edit this list after looking at the columns above. Missing ones are reported, not fatal.
The columns that use the price and never should be use as features
1. price_quote_price_per_night
2. price_quote_total_price, price_quote_raw
3. estimated_revenue_l365d

In [5]:


CANDIDATES = [
    "id",
    "price", # target (Not use in the featres)
    "bathrooms",
    "accommodates", "bedrooms", "beds", 
    "bathrooms_text", # This value must be parse from text to a number  text ("1.5 baths" → 1.5; "Half-bath" → 0.5).
    "room_type", "property_type",
    "neighbourhood_cleansed", "latitude", "longitude",
    "review_scores_rating", "number_of_reviews",
    "host_is_superhost", "host_response_rate", "hosts_time_as_host_years",
    "minimum_nights", "availability_365", "instant_bookable",
    "amenities",
]

keep = [c for c in CANDIDATES if c in raw.columns]
missing = [c for c in CANDIDATES if c not in raw.columns]
print("Kept:", len(keep), "| Missing from this snapshot:", missing)

df = raw.select(keep).cache()
df.count()

Kept: 21 | Missing from this snapshot: []


77149

## 4. Null rate per column (%)
Empty strings count as null too.

In [6]:
null_pct = df.agg(*[
    F.round(100 * F.avg((F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")), 1).alias(c)
    for c in keep
]).toPandas().T.rename(columns={0: "null_%"}).sort_values("null_%", ascending=False)
print(null_pct.to_string())

                          null_%
bathrooms                  100.0
host_response_rate         100.0
instant_bookable           100.0
price                       43.6
review_scores_rating        20.0
beds                         3.2
minimum_nights               0.2
bathrooms_text               0.1
id                           0.0
bedrooms                     0.0
accommodates                 0.0
property_type                0.0
room_type                    0.0
longitude                    0.0
latitude                     0.0
neighbourhood_cleansed       0.0
number_of_reviews            0.0
hosts_time_as_host_years     0.0
host_is_superhost            0.0
availability_365             0.0
amenities                    0.0


# According to the last analysis:
The variables that are useless in our analysis is: 
1. bathrooms
2. host_response_rate
3. instant_bookable

The price is a variable that has the *43.6%* of null, then around that quantity will not be use.

review_scores_rating => "review_scores_rating is missing at random (structurally: no reviews yet), so we gonna use the median imputation with a missing indicator (has_reviews)."                                                Explain in: https://arxiv.org/abs/2211.09259 

to the others:
1.  beds  =>  median imputation without a flag is enough
2.  minimum_nights   => median imputation without a flag is enough
3.  bathrooms_text =>  median imputation without a flag is enough         


## 5. Raw values of the tricky columns
I'ts important look at the formats to decide the cleaning rules.

In [7]:
from IPython.display import display

tricky = [c for c in ["price", "bathrooms_text", "host_is_superhost", "review_scores_rating",
                      "minimum_nights", "bedrooms", "beds", "accommodates",
                      "calculated_host_listings_count", "hosts_time_as_host_years",
                      "latitude", "longitude"] if c in keep]

# One row per column, one column per sample listing
sample = df.select(tricky).limit(9).toPandas().T
sample.columns = [f"row {i+1}" for i in range(sample.shape[1])]
display(sample)

if "amenities" in keep:
    am = (df.select(F.from_json("amenities", "array<string>").alias("arr"))
            .select(F.size("arr").alias("n_amenities"),
                    F.concat_ws(", ", F.slice("arr", 1, 6)).alias("first_6_amenities"))
            .limit(3).toPandas())
    display(am.style.set_properties(**{"text-align": "left"}))

,row 1,row 2,row 3,row 4,row 5,row 6,row 7,row 8,row 9
price,$272.50,$149.40,$177.75,$222.86,$57.53,$153.50,$724.00,None,$237.00
bathrooms_text,1 bath,1 bath,1 bath,1 bath,1 bath,1 bath,1 bath,1 bath,1.5 baths
host_is_superhost,f,f,f,f,t,f,f,f,f
review_scores_rating,4.9,4.72,4.92,4.92,4.95,4.85,4.83,4.4,5.0
minimum_nights,2,10,4,7,30,4,1,365,7
bedrooms,1,2,1,1,0,1,2,1,1
beds,1,1,1,1,1,1,3,1,1
accommodates,2,4,2,2,1,2,4,2,3
hosts_time_as_host_years,13,15,15,15,14,15,8,13,15
latitude,48.83191,48.85909,48.86373,48.88494,48.86441,48.82524108886719,48.86159,48.88259,48.88279


,n_amenities,first_6_amenities
0,15,"Iron, Children’s books and toys, Smoke alarm, Hangers, Hot water, Cooking basics"
1,25,"Free washer – In unit, Hot water, Refrigerator, Wifi, Dedicated workspace, Kitchen"
2,37,"Hot water, Refrigerator, Wifi, Dedicated workspace, Cleaning products, Kitchen"


## 6. Quick look at the price (target)

In [8]:
priced = df.withColumn("price_num", F.regexp_replace("price", "[$,€\\s]", "").cast("double"))

n_price = priced.filter(F.col("price_num").isNotNull()).count()
print(f"Rows with a usable price: {n_price:,} of {n_rows:,} ({100 * n_price / n_rows:.1f}%)")

priced.select("price_num").summary("count", "min", "25%", "50%", "75%", "max").show()

q = priced.approxQuantile("price_num", [0.01, 0.05, 0.95, 0.99], 0.001)
print("Percentiles 1% / 5% / 95% / 99%:", q)

Rows with a usable price: 43,537 of 77,149 (56.4%)


+-------+---------+
|summary|price_num|
+-------+---------+
|  count|    43537|
|    min|    10.38|
|    25%|    139.0|
|    50%|    218.0|
|    75%|    359.0|
|    max| 100666.1|
+-------+---------+



Percentiles 1% / 5% / 95% / 99%: [46.28, 70.0, 849.55, 1786.67]


## 7. Categorical columns: how many values?

In [9]:
for c in ["room_type", "neighbourhood_cleansed", "property_type", "host_is_superhost", "bathrooms_text"]:
    if c in keep:
        print(f"\n{c}: {df.select(c).distinct().count()} distinct values")
        df.groupBy(c).count().orderBy(F.desc("count")).show(25, truncate=False)


room_type: 4 distinct values


+---------------+-----+
|room_type      |count|
+---------------+-----+
|Entire home/apt|68333|
|Private room   |8280 |
|Hotel room     |386  |
|Shared room    |150  |
+---------------+-----+




neighbourhood_cleansed: 20 distinct values


+----------------------+-----+
|neighbourhood_cleansed|count|
+----------------------+-----+
|Buttes-Montmartre     |8064 |
|Popincourt            |6898 |
|Vaugirard             |6163 |
|Entrepôt              |5291 |
|Batignolles-Monceau   |5075 |
|Passy                 |4700 |
|Buttes-Chaumont       |4195 |
|Ménilmontant          |4075 |
|Opéra                 |3712 |
|Temple                |3474 |
|Reuilly               |3131 |
|Bourse                |2868 |
|Observatoire          |2748 |
|Élysée                |2604 |
|Panthéon              |2589 |
|Hôtel-de-Ville        |2501 |
|Gobelins              |2499 |
|Luxembourg            |2405 |
|Palais-Bourbon        |2255 |
|Louvre                |1902 |
+----------------------+-----+


property_type: 60 distinct values


+---------------------------------+-----+
|property_type                    |count|
+---------------------------------+-----+
|Entire rental unit               |64222|
|Private room in rental unit      |4986 |
|Entire condo                     |2057 |
|Room in hotel                    |1801 |
|Entire loft                      |832  |
|Private room in bed and breakfast|580  |
|Room in boutique hotel           |565  |
|Entire home                      |488  |
|Private room in condo            |289  |
|Entire serviced apartment        |245  |
|Entire townhouse                 |233  |
|Private room in home             |111  |
|Shared room in rental unit       |89   |
|Room in aparthotel               |86   |
|Private room in townhouse        |69   |
|Private room in loft             |59   |
|Private room in hostel           |45   |
|Entire place                     |43   |
|Shared room in hotel             |34   |
|Private room                     |32   |
|Private room in guest suite      


host_is_superhost: 2 distinct values


+-----------------+-----+
|host_is_superhost|count|
+-----------------+-----+
|f                |59655|
|t                |17494|
+-----------------+-----+




bathrooms_text: 37 distinct values
+-----------------+-----+
|bathrooms_text   |count|
+-----------------+-----+
|1 bath           |55438|
|2 baths          |5547 |
|1.5 baths        |5299 |
|1 private bath   |4032 |
|1 shared bath    |2556 |
|2.5 baths        |1395 |
|3 baths          |945  |
|0 baths          |413  |
|Half-bath        |256  |
|1.5 shared baths |236  |
|3.5 baths        |223  |
|4 baths          |189  |
|Shared half-bath |142  |
|2 shared baths   |91   |
|Private half-bath|83   |
|0 shared baths   |65   |
|4.5 baths        |61   |
|5 baths          |53   |
|NULL             |49   |
|6 baths          |21   |
|5.5 baths        |14   |
|3 shared baths   |11   |
|7 baths          |6    |
|2.5 shared baths |5    |
|6.5 baths        |4    |
+-----------------+-----+
only showing top 25 rows


## 8. Create the 5,000-row sample
We kept columns, still raw (not cleaned), so the same cleaning code works on both files.
Only rows **with a price** are sampled: rows without the target are dropped by cleaning anyway
(43.6% of the raw file), so keeping them would waste almost half of the sample.
Written with pandas to avoid Spark's winutils problem on Windows.

In [ ]:
# Keep only rows with a price that is not null or ""
priced_raw = df.filter(F.col("price").isNotNull() & (F.trim(F.col("price")) != ""))
n_priced = priced_raw.count()

# Sample of at least 5000 rows
fraction = min(1.0, SAMPLE_SIZE * 1.2 / n_priced)
sample_pdf = priced_raw.sample(fraction=fraction, seed=SEED).limit(SAMPLE_SIZE).toPandas()

os.makedirs(SAMPLE_DIR, exist_ok=True)
sample_path = os.path.join(SAMPLE_DIR, "listings_sample.csv")
sample_pdf.to_csv(sample_path, index=False)
print(f"Sample written: {sample_path} ({len(sample_pdf):,} rows)")

# Check it reads back with the same options as the full file
check = (spark.read.option("header", True).option("multiLine", True)
         .option("escape", '"').csv(sample_path))
print("Re-read:", check.count(), "rows,", len(check.columns), "columns")
assert check.count() == SAMPLE_SIZE

[Stage 74:>                                                         (0 + 1) / 1]